# Clean, Source-Organized Copy
Saved outputs and machine-specific metadata are intentionally excluded. The byte-identical original remains in the local audit archive. Prepare the local workbench before running this notebook. Notebooks 01 and 02 execute GPU inference when explicitly enabled; 03 disables its optional GPU activation check by default.


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "scripts" / "lab.py").is_file()
)
WORKSPACE = PROJECT_ROOT / ".local" / "workbench"
if not (WORKSPACE / "weight_edit.py").is_file():
    raise RuntimeError("Run python scripts/lab.py prepare --with-local-data first.")
os.chdir(WORKSPACE)
sys.path.insert(0, str(WORKSPACE))


# What does the edit actually change?

This notebook studies the **geometry** of the published correction before making claims about behavior. It reads only the two verified local weight matrices, computes their differences in FP32, and visualizes the changed columns and singular directions. No training, downloads, or extra plotting packages are required.

Select **Python (weight-space)**. The first part is CPU-only. The optional second part loads the model on the GPU for four short, teacher-forced forward passes.

In [ ]:
from IPython.display import SVG, display
import torch

from edit_anatomy import (
    TEACHER_TEXTS, analyze_edit, capture_teacher_projections,
    save_anatomy, save_projections,
)
from experiment_runner import load_local_model, new_run_directory, verify_base

RUN_TEACHER_FORCING = False

In [ ]:
anatomy = analyze_edit()
output_dir = new_run_directory('anatomy')
plots = save_anatomy(anatomy, output_dir)

for row in anatomy['layers']:
    print(f"Layer {row['layer']}: {row['changed_entries']:,} values / {row['changed_columns']} columns")
    print(f"  Delta norm / original matrix norm: {row['relative_frobenius_change']:.3%}")
    print(f"  Fraction of delta energy in the first direction: {row['top_direction_energy']:.5%}")
    print('  Singular components needed for 99% energy:', row['rank_for_99_percent_energy'])
    print('  First twelve actual changed-column indices:', row['column_indices'][:12])
print('Absolute cosine between the two leading directions:', anatomy['direction_absolute_cosine'])
print('Results:', output_dir.relative_to(output_dir.parents[1]))

## Read the plots

- A first singular component carrying almost all delta energy means the edit is approximately one-directional, not thousands of unrelated changes.
- A high absolute cosine means the two layers' leading corrections point along similar residual-space directions. Singular-vector sign is arbitrary.
- The column plots use actual MLP column indices. Heatmaps use the ordinal positions of the 64 selected columns, with each row averaging 16 residual dimensions. Colors are normalized separately for each heatmap.

**These are mathematical descriptions, not proof that the direction means 'looping'.** Any interpretation must survive behavioral controls.

In [ ]:
for path in plots:
    display(SVG(filename=str(path)))

## Same-input activation check

Now compare the MLP output's projection onto the leading edit direction, using **identical fixed text** in both conditions. One text is a short calculation; the other repeats a rechecking sentence. This avoids comparing different generated token histories.

These are synthetic teacher-forced inputs, not naturally occurring model loops. A projection change shows that the edit mechanically affects these activations; it does not establish a semantic or causal explanation for all looping behavior.

Set `RUN_TEACHER_FORCING = False` to skip GPU work. Shut down other model kernels if memory is limited.

In [ ]:
if RUN_TEACHER_FORCING:
    model, tokenizer = load_local_model()
    try:
        projections = capture_teacher_projections(model, tokenizer, anatomy)
        verify_base(model)
        projection_plots = save_projections(projections, output_dir)
        for row in projections:
            for layer, values in row['layers'].items():
                mean = sum(values['absolute_cosine']) / len(values['absolute_cosine'])
                print(row['condition'], row['text_id'], 'layer', layer, 'mean |cosine|:', round(mean, 5))
        for path in projection_plots:
            display(SVG(filename=str(path)))
        print('Both original tensor hashes verified after the forward passes.')
    finally:
        del model
        torch.cuda.empty_cache()
else:
    print('Teacher-forced GPU check skipped.')

## What to investigate next

The low-rank structure motivates the half-strength and single-layer controls in `04_controlled_experiments.ipynb`. Do not infer improvement from a convincing-looking heatmap. The controlled pilot checks delivered answers, natural stopping, repetition, and cost against both thinking-on and thinking-off baselines.

All figures and scalar projection traces stay in the timestamped local results directory. The checkpoint is never rewritten.